# Capítulo 5 — Hidrodinâmica de Canais Abertos

**Autor:** Jader Lugon Junior  
**Livro:** Fenômenos de Transporte: Fundamentos e Modelagem Computacional  
**Repositório:** [github.com/JaderLugon/fenomenos-transporte-livro](https://github.com/JaderLugon/fenomenos-transporte-livro)

---

## 🎯 Objetivos de Aprendizagem

Ao final deste notebook, você será capaz de:

1. **Derivar** as equações de Saint-Venant 1D a partir dos princípios de conservação
2. **Calcular** a profundidade normal em canais retangulares e trapezoidais usando Manning
3. **Interpretar** o número de Froude e classificar regimes (subcrítico, crítico, supercrítico)
4. **Aplicar** a condição CFL para estabilidade de esquemas explícitos
5. **Implementar** um solver numérico para simular a propagação de ondas de cheia
6. **Analisar** a celeridade da onda gravitacional e o tempo de trânsito em rios

---

## 📚 Conteúdo Teórico Resumido

### 5.1 Equações de Saint-Venant

As equações de águas rasas governam escoamentos com superfície livre em canais e rios:

$$
\frac{\partial h}{\partial t} + \frac{\partial (uh)}{\partial x} = 0
$$

$$
\frac{\partial u}{\partial t} + u \frac{\partial u}{\partial x} + g \frac{\partial h}{\partial x} = g(S_0 - S_f)
$$

### 5.2 Declividade de Atrito (Manning)

$$
S_f = \frac{n^2 \, u \, |u|}{h^{4/3}}
$$

### 5.3 Celeridade da Onda Gravitacional

$$
c = \sqrt{g \, h}
$$

### 5.4 Número de Froude

$$
\mathrm{Fr} = \frac{u}{\sqrt{g \, h}}
$$

- $\mathrm{Fr} < 1$: regime **subcrítico** (fluvial)
- $\mathrm{Fr} = 1$: regime **crítico**
- $\mathrm{Fr} > 1$: regime **supercrítico** (torrencial)

### 5.5 Condição CFL para Esquemas Explícitos

$$
\mathrm{CFL} = \frac{(|u| + \sqrt{gh}) \, \Delta t}{\Delta x} \leq C_{\max} \approx 0,8
$$

---

In [ ]:
# ============================================================
# CONFIGURAÇÃO INICIAL
# ============================================================
import sys
import os

# Adiciona o diretório raiz ao path para importar módulos
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import brentq

# Configuração de plots
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

print("✅ Ambiente configurado com sucesso!")
print(f"📦 NumPy {np.__version__} | Matplotlib {plt.matplotlib.__version__}")

---

## 🔬 Seção 1: Profundidade Normal (Manning)

### Exercício 1.1: Profundidade Normal em Canal Retangular

Calcule a profundidade normal $h_n$ de um canal retangular com:
- Largura $B = 42,2$ m
- Rugosidade de Manning $n = 0,035$ s/m$^{1/3}$
- Declividade $S_0 = 1,0 \times 10^{-4}$
- Vazão $Q = 6,7$ m³/s

**Solução:** Para canal retangular largo, $R_h \approx h$ e $A = B \cdot h$. A equação de Manning é:

$$
Q = \frac{1}{n} \, B \, h \, h^{2/3} \, S_0^{1/2} = \frac{B}{n} \, h^{5/3} \, S_0^{1/2}
$$

Isolando $h$:

$$
h = \left(\frac{Q \, n}{B \, S_0^{1/2}}\right)^{3/5}
$$

In [ ]:
# ============================================================
# EXERCÍCIO 1.1: Profundidade Normal (Manning)
# ============================================================

print("=" * 60)
print("EXERCÍCIO 1.1: PROFUNDIDADE NORMAL EM CANAL RETANGULAR")
print("=" * 60)

# Dados do problema
B = 42.2        # m (largura)
n = 0.035       # s/m^(1/3) (Manning)
S0 = 1.0e-4     # adimensional (declividade)
Q = 6.7         # m³/s (vazão)

# Cálculo da profundidade normal
h_n = (Q * n / (B * np.sqrt(S0)))**(3/5)

# Verificação: vazão calculada pela equação de Manning
Q_Manning = (B / n) * h_n**(5/3) * np.sqrt(S0)

print(f"\n📊 Dados do problema:")
print(f"  • Largura: B = {B} m")
print(f"  • Rugosidade: n = {n} s/m^(1/3)")
print(f"  • Declividade: S₀ = {S0:.1e}")
print(f"  • Vazão: Q = {Q} m³/s")

print(f"\n🧮 Cálculo:")
print(f"  • h_n = (Q·n / (B·√S₀))^(3/5)")
print(f"  • h_n = ({Q} × {n} / ({B} × {np.sqrt(S0):.4f}))^(3/5)")
print(f"  • h_n = {h_n:.4f} m")

print(f"\n✅ Verificação:")
print(f"  • Q_Manning = {Q_Manning:.3f} m³/s")
print(f"  • Erro relativo: {abs(Q_Manning - Q)/Q * 100:.2f}%")

print(f"\n💡 Interpretação:")
print(f"  A profundidade normal de {h_n:.2f} m é consistente com")
print(f"  um rio de planície de baixa declividade e vazão moderada.")

---

## 🌊 Seção 2: Número de Froude e Classificação de Regimes

### Exercício 2.1: Classificação do Rio Macaé

Calcule o número de Froude para o Rio Macaé com:
- Velocidade média $U = 0,224$ m/s
- Profundidade $h = 0,71$ m
- $g = 9,81$ m/s²

In [ ]:
# ============================================================
# EXERCÍCIO 2.1: Número de Froude
# ============================================================

print("=" * 60)
print("EXERCÍCIO 2.1: NÚMERO DE FROUDE - RIO MACAÉ")
print("=" * 60)

# Dados
U = 0.224       # m/s
h = 0.71        # m
g = 9.81        # m/s²

# Cálculo
c = np.sqrt(g * h)  # celeridade da onda gravitacional
Fr = U / c

# Classificação
if Fr < 0.95:
    regime = "SUBCRÍTICO (fluvial)"
elif Fr > 1.05:
    regime = "SUPERCRÍTICO (torrencial)"
else:
    regime = "CRÍTICO"

print(f"\n📊 Dados do problema:")
print(f"  • Velocidade média: U = {U} m/s")
print(f"  • Profundidade: h = {h} m")
print(f"  • Gravidade: g = {g} m/s²")

print(f"\n🧮 Cálculo:")
print(f"  • Celeridade: c = √(g·h) = √({g} × {h}) = {c:.3f} m/s")
print(f"  • Fr = U/c = {U}/{c:.3f} = {Fr:.3f}")

print(f"\n🏷️ Classificação: {regime}")

print(f"\n💡 Interpretação:")
print(f"  • Fr < 1 → regime FLUVIAL: ondas gravitacionais podem")
print(f"    se propagar contra a correnteza (a montante)")
print(f"  • A pressão hidrostática é uma hipótese válida")
print(f"  • Equações de Saint-Venant são aplicáveis")
print(f"  • Perturbações a jusante afetam o escoamento a montante")

---

## ⚡ Seção 3: Condição CFL e Estabilidade Numérica

### Exercício 3.1: Passo de Tempo Máximo

Calcule o maior $\Delta t$ permitido pela condição CFL para:
- $h = 0,71$ m, $u = 0,224$ m/s
- $\Delta x = 100$ m
- $C_{\max} = 0,8$

**Solução:**

$$
\Delta t_{\max} = \frac{C_{\max} \cdot \Delta x}{|u| + \sqrt{gh}}
$$

In [ ]:
# ============================================================
# EXERCÍCIO 3.1: Condição CFL
# ============================================================

print("=" * 60)
print("EXERCÍCIO 3.1: CONDIÇÃO CFL - PASSO DE TEMPO MÁXIMO")
print("=" * 60)

# Dados
u = 0.224       # m/s
h = 0.71        # m
dx = 100.0      # m
Cmax = 0.8      # adimensional
g = 9.81        # m/s²

# Cálculo
c = np.sqrt(g * h)
v_max = abs(u) + c
dt_max = Cmax * dx / v_max
dt_recomendado = 0.7 * dt_max

print(f"\n📊 Dados do problema:")
print(f"  • Velocidade: u = {u} m/s")
print(f"  • Profundidade: h = {h} m")
print(f"  • Espaçamento: Δx = {dx} m")
print(f"  • CFL máximo: Cmax = {Cmax}")

print(f"\n🧮 Cálculo:")
print(f"  • Celeridade: c = √(g·h) = {c:.3f} m/s")
print(f"  • Velocidade máxima: |u| + c = {v_max:.3f} m/s")
print(f"  • Δt_max = Cmax·Δx/(|u|+c) = {Cmax}×{dx}/{v_max:.3f}")
print(f"  • Δt_max = {dt_max:.2f} s")

print(f"\n✅ Recomendação:")
print(f"  • Δt recomendado (70% do máximo): {dt_recomendado:.2f} s")
print(f"  • Adotaremos Δt = 30 s (valor comercial)")

print(f"\n💡 Interpretação:")
print(f"  A condição CFL garante que a informação não 'pule'")
print(f"  mais de uma célula por passo de tempo. Violá-la causa")
print(f"  crescimento exponencial de erros (explosão numérica).")

---

## 🌪️ Seção 4: Celeridade e Tempo de Trânsito

### Exercício 4.1: Tempo de Viagem de uma Onda de Cheia

Para um trecho de 6,3 km do Rio Macaé com celeridade $c \approx 2,64$ m/s, estime o tempo de trânsito do pico de cheia.

In [ ]:
# ============================================================
# EXERCÍCIO 4.1: Tempo de Trânsito
# ============================================================

print("=" * 60)
print("EXERCÍCIO 4.1: TEMPO DE TRÂNSITO DE ONDA DE CHEIA")
print("=" * 60)

# Dados
L = 6300.0      # m (comprimento do trecho)
c = 2.64        # m/s (celeridade da onda)
u = 0.224       # m/s (velocidade do escoamento)

# Cálculo
v_onda = u + c  # velocidade efetiva de propagação
t_transito = L / v_onda
t_transito_min = t_transito / 60

print(f"\n📊 Dados do problema:")
print(f"  • Comprimento do trecho: L = {L/1000:.1f} km")
print(f"  • Celeridade: c = {c} m/s")
print(f"  • Velocidade do escoamento: u = {u} m/s")

print(f"\n🧮 Cálculo:")
print(f"  • Velocidade efetiva: v = u + c = {u} + {c} = {v_onda:.3f} m/s")
print(f"  • Tempo de trânsito: t = L/v = {L}/{v_onda:.3f}")
print(f"  • t = {t_transito:.0f} s ≈ {t_transito_min:.1f} min")

print(f"\n💡 Interpretação prática:")
print(f"  O pico de cheia leva ~{t_transito_min:.0f} minutos para percorrer")
print(f"  os {L/1000:.1f} km do domínio. Esse tempo é fundamental para:")
print(f"    • Sistemas de alerta de cheias")
print(f"    • Planejamento de evacuação")
print(f"    • Operação de reservatórios")

---

## 📊 Seção 5: Visualização do Perfil de Velocidade

### Exercício 5.1: Perfil de Velocidade em Canal Retangular

Plote o perfil de velocidade (logarítmico) em um canal retangular com $h = 0,71$ m, $u_* = 0,026$ m/s e $n = 0,035$.

In [ ]:
# ============================================================
# EXERCÍCIO 5.1: Perfil de Velocidade Logarítmico
# ============================================================

# Parâmetros
h = 0.71        # m
u_star = 0.026  # m/s (velocidade de atrito)
kappa = 0.41    # constante de von Kármán
nu = 1.0e-6     # m²/s (viscosidade cinemática da água)

# Perfil vertical (z de 0 a h)
z = np.linspace(0.001, h, 100)
z_plus = z * u_star / nu

# Lei da parede (regime turbulento)
u_log = (u_star / kappa) * np.log(z_plus) + 5.1 * u_star

# Perfil parabólico (laminar, para comparação)
u_max_lam = 1.5 * 0.224  # velocidade média × 1.5
u_parab = u_max_lam * (1 - (z/h)**2)

# Plot
fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(u_log, z, 'b-', linewidth=2, label='Perfil logarítmico (turbulento)')
ax.plot(u_parab, z, 'r--', linewidth=2, label='Perfil parabólico (laminar)')
ax.set_xlabel('Velocidade, u [m/s]', fontsize=12)
ax.set_ylabel('Profundidade, z [m]', fontsize=12)
ax.set_title('Perfil de Velocidade Vertical em Canal', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\n💡 Observação:")
print("  O perfil logarítmico (turbulento) é mais 'achatado' próximo")
print("  à superfície e apresenta gradiente muito acentuado junto ao fundo.")
print("  Isso reflete a intensa mistura turbulenta no escoamento real.")

---

## 🧮 Seção 6: Solver Explícito Simplificado

### Exercício 6.1: Propagação de Degrau de Vazão

Implemente um esquema explícito simplificado para simular a propagação de um degrau de vazão em um canal retangular.

In [ ]:
# ============================================================
# EXERCÍCIO 6.1: Solver Explícito Simplificado
# ============================================================

print("=" * 60)
print("EXERCÍCIO 6.1: PROPAGAÇÃO DE DEGRAU DE VAZÃO")
print("=" * 60)

# Parâmetros do canal
B = 42.2        # m (largura)
n = 0.035       # s/m^(1/3)
S0 = 1.0e-4     # declividade
g = 9.81        # m/s²

# Discretização
L = 6300.0      # m
N = 63          # número de células
dx = L / N      # m
dt = 30.0       # s

# Condição inicial: regime uniforme
Q_ref = 6.7     # m³/s
h0 = (Q_ref * n / (B * np.sqrt(S0)))**(3/5)

# Arrays
h = np.full(N, h0)
u = np.full(N, Q_ref / (B * h0))

# Hidrograma de entrada (degrau)
def Q_in(t):
    if t < 3600:
        return Q_ref
    elif t < 7200:
        return Q_ref + (12.0 - Q_ref) * (t - 3600) / 3600
    else:
        return 12.0

# Loop temporal
t_final = 7200.0  # 2 horas
t = 0.0
resultados = []

while t < t_final:
    # Contorno de montante
    u[0] = Q_in(t) / (B * h[0])
    
    # Atualização explícita (onda cinemática simplificada)
    for i in range(1, N):
        # Conservação de massa
        Q_atual = u[i] * B * h[i]
        Q_anterior = u[i-1] * B * h[i-1]
        h[i] = h[i] - (dt / dx) * (Q_atual - Q_anterior) / B
        
        # Manning para atualizar velocidade
        if h[i] > 0.01:
            u[i] = (1/n) * h[i]**(2/3) * np.sqrt(S0)
    
    # Contorno de jusante (onda saindo)
    h[-1] = h[-2]
    
    t += dt
    if t % 1800 == 0:  # salvar a cada 30 min
        resultados.append((t/3600, h.copy()))

# Plot dos resultados
fig, ax = plt.subplots(figsize=(10, 6))
x = np.linspace(0, L/1000, N)

for t_h, h_perf in resultados:
    ax.plot(x, h_perf, linewidth=2, label=f't = {t_h:.1f} h')

ax.set_xlabel('Distância, x [km]', fontsize=12)
ax.set_ylabel('Profundidade, h [m]', fontsize=12)
ax.set_title('Propagação de Onda de Cheia no Rio Macaé', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\n✅ Simulação concluída!")
print(f"  • Profundidade inicial: {h0:.3f} m")
print(f"  • Profundidade máxima (pico): ~{max(h):.3f} m")
print(f"  • Atenuação do pico: ~{(1 - max(h)/((12.0*n/(B*np.sqrt(S0)))**(3/5)))*100:.1f}%")

---

## 🔬 Estudos de Caso

Os estudos de caso aplicam os conceitos deste capítulo em problemas reais de engenharia.

| Estudo de Caso | Descrição | Link |
|----------------|-----------|------|
| **Caso 5.1** | Hidrodinâmica do Rio Macaé | [🔗 Abrir](../casos/05_1_Rio_Macae_Hidrodinamica.ipynb) |
| **Caso 5.2** | Profundidade Normal em Canal Trapezoidal | [🔗 Abrir](../casos/05_2_Profundidade_Normal_Canal_Trapezoidal.ipynb) |

> 💡 **Dica:** Os estudos de caso podem ser executados independentemente deste notebook principal.

---

## 📖 Referências

- CHOW, V. T. *Open-Channel Hydraulics*. McGraw-Hill, 1959.
- MALISKA, C. R. *Transferência de Calor e Mecânica dos Fluidos Computacional*. 2ª ed. LTC, 2004.
- LUGON JR., J. et al. (2008). Assessment of dispersion mechanisms in rivers by means of an inverse problem approach. *Inverse Problems in Science and Engineering*, 16(8), 967–979.
- MOUKALLED, F.; MANGANI, L.; DARWISH, M. *The Finite Volume Method in Computational Dynamics*. Springer, 2016.

---

## 🔄 Navegação

- [📚 Capítulo Anterior: Escoamento em Tubulações](04_Escoamento_Tubulacoes.ipynb)
- [📚 Próximo Capítulo: Percolação em Meio Poroso](06_Percolacao_Meio_Poroso.ipynb)
- [📂 Outros Capítulos](../notebooks/)
- [🏠 Repositório Principal](../README.md)

---

<div align="center">

**QR Code do Capítulo 5**  
Aponte seu celular para o QR Code no livro para acessar este notebook!

</div>

In [ ]:
print("=" * 60)
print("✅ NOTEBOOK CONCLUÍDO COM SUCESSO!")
print("=" * 60)
print("\n🎓 Você completou o Capítulo 5!")
print("📖 Próximo passo: Capítulo 6 - Percolação em Meio Poroso")
print("\n💡 Dica: Execute este notebook quantas vezes precisar!")
print("   Modifique os parâmetros e explore diferentes cenários.")